Objectives:

- link between relative frequencies and probabilities

- conditional probabilities and the law of total probability

Data file: nlsy97_transitions.parquet

on Moodle: Employment Status and College Enrollment Transitions



#### Loading the packages

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


#### Loading the data

In [3]:
ROOT = 'c:/Users/PC_DS_ECON_5/Desktop/data-analytics-python-example'
DATA = ROOT + '/data'


In [4]:
df = pd.read_parquet(DATA + 
                     '/nlsy97_transitions.parquet')

Structure of the dataset is very similar to nlsy97_emp_hed.parquet

Only differences: 

 - two new variables

 - if the two new variables are missing, then the observation was removed

Recall that the unit of observation was:

`person_id` X `year_month`


In [ ]:
# new variables are never missing
df[['emp_category_next', 'college_status_next']].isna().any().sum()

np.int64(0)

In [ ]:
# unit of observation is individual X month
df[['person_id', 'year_month']].value_counts().value_counts()

count
1    2111795
Name: count, dtype: int64

Let us work with observations from a single month

Let us select a month randomly

`numpy` package has a "random number generator" 

 - pseudo-random number generator
 - one specifies a particular number (seed number)
 - every time one runs the same code, the same "random" values are selected
 - when specifying the seed number, we do not know what random draws will occur
 - but they are the same if code is run from the beginning

Creating a random number generator object with "seed" value

rng abbreviation for random number generator

Function: np.random.default_rng()



In [11]:
rng = np.random.default_rng(seed=8763534)

Objective: let us select a month randomly 

using our random number generator and `choice()` function attached to it 

First:
list of months (each only once), and not the entire column of the dataset
such that every month has the same probability of being drawn randomly

In [16]:
# running the code from the beginning 
# (from the specification of the seed value)
# will always produce the same result
rng = np.random.default_rng(seed=8763534)
chosen_month = rng.choice(
    # set of values from which to choose "randomly"
    a=df['year_month'].drop_duplicates()
)
chosen_month

Period('2020-08', 'M')

Advantage: reproduceable

Point to keep in mind:
seed value can be / is chosen arbitrarily
I can choose the seed value that produces the result that I want.

Let us subset the dataset, let us treat the observations from the chosen month as a *statistical population*. Statistical population, because the objective then is to draw random samples from this statistical population.

In [17]:
# keeping only observations from the chosen month
df1 = (
    df
        .loc[df['year_month'].eq(chosen_month),:]
        .reset_index(drop=True)
        .copy()
)

In [18]:
df1['year_month'].drop_duplicates()

0    2020-08
Name: year_month, dtype: period[M]

Let us create a variable:

a dummy variable

non-participant =
1 if the person is non-participant (`emp_category=non-participant`)
0 otherwise

binary variable, with two values 1 and 0 (answer to a Yes/No question)


In [19]:
# numpy.where(condition, value if true, value if false)
# conditional assignment of values to variables
df1['non-participant'] = np.where(
    df1['emp_category'].eq('non-participant'),
    1,
    0    
)

`df1` is the statistical population (thought experiment)

for this population, let us compute 
 - the relative frequency/share/proportion of being non-participant 

number of individuals with non-participant=1
divided by
number of individuals alltogether


In [22]:
population_rel_freq = (
    # number of individuals who are non-participant
    df1['non-participant'].eq(1).sum()
    # divided by
    /
    # total number of individuals
    df1.shape[0]
)

In [23]:
population_rel_freq

np.float64(0.16533864541832669)

The objective from now on will be to draw random samples from this population and compare the relative frequencies of non-participants with the population relative frequency.

Illustrate a version of the law of large numbers.
What happens as the sample size increases?